# YgoMaster-PackBuilder — YDK → Master Duel ID Mapping Tests

Notebook di test manuale/esplorativo per il resolver:

```text
YdkIds.txt
    ↓
load_ydk_mapping(...)
    ↓
dict[ydk_id, master_duel_id]
    ↓
resolve_card_master_duel_id(...)
    ↓
Card.master_duel_id
```

Il notebook contiene:

1. test **offline** con file temporanei;
2. test di risoluzione su `Card`;
3. raccolta delle carte irrisolte;
4. una sezione **opzionale** per testare il vero `YgoMaster/Data/YdkIds.txt`.

I test offline non richiedono una installazione YgoMaster presente nella repository.


## 1. Ambiente e root repository

In [19]:
from pathlib import Path
import sys

print("Python interpreter:", sys.executable)
print("Current working directory:", Path.cwd())

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    repo_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError(
        "Impossibile individuare la root della repository."
    )

print("Repository root:", repo_root)

Python interpreter: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\.venv\Scripts\python.exe
Current working directory: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\tests
Repository root: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder


## 2. Import

In [20]:
from tempfile import TemporaryDirectory

from ygomaster_packbuilder.domain import Card, Rarity

from ygomaster_packbuilder.mapping import (
    load_ydk_mapping,
    resolve_card_master_duel_id,
    resolve_cards_master_duel_ids,
)

print("Imports OK")

Imports OK


## 3. File `YdkIds.txt` temporaneo valido

In [21]:
sample_text = '''
34541863 7128
91123920 12345
11111111 500
22222222 500
'''.strip()

with TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "YdkIds.txt"
    path.write_text(sample_text, encoding="utf-8")

    mapping = load_ydk_mapping(path)

mapping

{34541863: 7128, 91123920: 12345, 11111111: 500, 22222222: 500}

## 4. Controllo del mapping caricato

In [22]:
assert isinstance(mapping, dict)

assert mapping[34541863] == 7128
assert mapping[91123920] == 12345
assert mapping[11111111] == 500
assert mapping[22222222] == 500

for ydk_id, master_duel_id in mapping.items():
    assert isinstance(ydk_id, int)
    assert isinstance(master_duel_id, int)

print("YDK mapping loaded correctly")

YDK mapping loaded correctly


## 5. Più YDK ID possono puntare allo stesso Master Duel ID

Il mapping deve essere:

```text
YDK → Master Duel
```

ma non deve necessariamente essere biunivoco.


In [23]:
assert mapping[11111111] == mapping[22222222] == 500

print("Many-to-one mappings are accepted")

Many-to-one mappings are accepted


## 6. Risoluzione di una singola carta

In [24]:
card = Card(
    name='"A" Cell Breeding Device',
    ydk_id=34541863,
    source_rarity="Common",
    rarity=Rarity.N,
)

assert card.master_duel_id is None

resolved = resolve_card_master_duel_id(
    card,
    mapping,
)

assert resolved is True
assert card.master_duel_id == 7128

print(card)

Card(name='"A" Cell Breeding Device', ydk_id=34541863, source_rarity='Common', master_duel_id=7128, rarity=<Rarity.N: 'N'>)


## 7. Carta non presente nel mapping

Una carta irrisolta non deve causare un'eccezione.

La funzione deve restituire `False` e lasciare:

```text
master_duel_id = None
```


In [25]:
card = Card(
    name="Unknown Card",
    ydk_id=99999999,
    source_rarity="Common",
    rarity=Rarity.N,
)

resolved = resolve_card_master_duel_id(
    card,
    mapping,
)

assert resolved is False
assert card.master_duel_id is None

print("Unresolved card handled correctly")

Unresolved card handled correctly


## 8. Risoluzione di una lista di carte

In [26]:
cards = [
    Card(
        name="Resolved A",
        ydk_id=34541863,
        source_rarity="Common",
        rarity=Rarity.N,
    ),
    Card(
        name="Missing",
        ydk_id=99999999,
        source_rarity="Rare",
        rarity=Rarity.R,
    ),
    Card(
        name="Resolved B",
        ydk_id=91123920,
        source_rarity="Super Rare",
        rarity=Rarity.SR,
    ),
]

unresolved = resolve_cards_master_duel_ids(
    cards,
    mapping,
)

assert cards[0].master_duel_id == 7128
assert cards[1].master_duel_id is None
assert cards[2].master_duel_id == 12345

assert len(unresolved) == 1
assert unresolved[0] is cards[1]

print("Resolved:")
for card in cards:
    print(card.name, "->", card.master_duel_id)

print()
print("Unresolved:", [card.name for card in unresolved])

Resolved:
Resolved A -> 7128
Missing -> None
Resolved B -> 12345

Unresolved: ['Missing']


## 9. Il resolver non modifica rarità o metadata

Il resolver deve modificare esclusivamente `master_duel_id`.


In [27]:
card = Card(
    name="Metadata Test",
    ydk_id=34541863,
    source_rarity="Ultra Rare",
    rarity=Rarity.UR,
)

original_name = card.name
original_source_rarity = card.source_rarity
original_rarity = card.rarity

resolve_card_master_duel_id(card, mapping)

assert card.name == original_name
assert card.source_rarity == original_source_rarity
assert card.rarity is original_rarity
assert card.master_duel_id == 7128

print("Only master_duel_id was modified")

Only master_duel_id was modified


## 10. File inesistente

In [28]:
missing = repo_root / "__missing_YdkIds.txt"

try:
    load_ydk_mapping(missing)
except FileNotFoundError as exc:
    print("Expected FileNotFoundError:", exc)
else:
    raise AssertionError(
        "Il loader avrebbe dovuto sollevare FileNotFoundError."
    )

Expected FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\LMDP1\\Desktop\\GitHub\\YgoMaster-PackBuilder\\__missing_YdkIds.txt'


## 11. Riga con numero errato di colonne

In [29]:
with TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "YdkIds.txt"

    path.write_text(
        "34541863 7128 extra\n",
        encoding="utf-8",
    )

    try:
        load_ydk_mapping(path)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError(
            "Il loader avrebbe dovuto rifiutare una riga con 3 valori."
        )

Expected ValueError: Invalid YdkIds.txt line 1: expected 2 values, found 3.


## 12. Valore non numerico

In [30]:
with TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "YdkIds.txt"

    path.write_text(
        "not-a-number 7128\n",
        encoding="utf-8",
    )

    try:
        load_ydk_mapping(path)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError(
            "Il loader avrebbe dovuto rifiutare un YDK ID non numerico."
        )

Expected ValueError: Invalid numeric value in YdkIds.txt at line 1.


## 13. YDK ID duplicato

In [31]:
with TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "YdkIds.txt"

    path.write_text(
        "34541863 7128\n"
        "34541863 9999\n",
        encoding="utf-8",
    )

    try:
        load_ydk_mapping(path)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError(
            "Il loader avrebbe dovuto rifiutare un YDK ID duplicato."
        )

Expected ValueError: Duplicate YDK ID 34541863 at line 2.


## 14. Righe vuote

In [32]:
with TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "YdkIds.txt"

    path.write_text(
        "\n34541863 7128\n\n91123920 12345\n\n",
        encoding="utf-8",
    )

    loaded = load_ydk_mapping(path)

assert loaded == {
    34541863: 7128,
    91123920: 12345,
}

print("Blank lines correctly ignored")

Blank lines correctly ignored


# Test opzionale sul vero YgoMaster

Le celle successive non devono fallire se non hai ancora impostato il percorso.

Imposta `YGOMASTER_ROOT` alla directory che contiene:

```text
YgoMaster/
└── Data/
    └── YdkIds.txt
```


## 15. Percorso reale di YgoMaster

In [33]:
# Esempio Windows:
# YGOMASTER_ROOT = Path(r"C:\path\to\YgoMaster")
#
# Esempio Linux:
# YGOMASTER_ROOT = Path("/path/to/YgoMaster")

YGOMASTER_ROOT: Path | None = Path("C:\\Program Files (x86)\\Steam\\steamapps\\common\\Yu-Gi-Oh!  Master Duel\\YgoMaster")

if YGOMASTER_ROOT is None:
    print("Real YgoMaster test disabled.")
else:
    real_ydk_path = YGOMASTER_ROOT / "Data" / "YdkIds.txt"
    print("YdkIds.txt:", real_ydk_path)
    assert real_ydk_path.exists(), real_ydk_path

YdkIds.txt: C:\Program Files (x86)\Steam\steamapps\common\Yu-Gi-Oh!  Master Duel\YgoMaster\Data\YdkIds.txt


## 16. Caricamento del mapping reale

In [34]:
if YGOMASTER_ROOT is None:
    print("Skipped: set YGOMASTER_ROOT to enable this test.")
else:
    real_ydk_path = YGOMASTER_ROOT / "Data" / "YdkIds.txt"
    real_mapping = load_ydk_mapping(real_ydk_path)

    print("Mappings loaded:", len(real_mapping))

    assert len(real_mapping) > 0

Mappings loaded: 17991


## 17. Carta nota nel mapping reale

Verifichiamo il caso già noto:

```text
"A" Cell Breeding Device
YDK: 34541863
Master Duel ID: 7128
```


In [35]:
if YGOMASTER_ROOT is None:
    print("Skipped: set YGOMASTER_ROOT to enable this test.")
else:
    assert 34541863 in real_mapping

    master_duel_id = real_mapping[34541863]

    print("34541863 ->", master_duel_id)

    assert master_duel_id == 7128

34541863 -> 7128


## 18. Risoluzione di una `Card` col mapping reale

In [36]:
if YGOMASTER_ROOT is None:
    print("Skipped: set YGOMASTER_ROOT to enable this test.")
else:
    card = Card(
        name='"A" Cell Breeding Device',
        ydk_id=34541863,
        source_rarity="Common",
        rarity=Rarity.N,
    )

    resolved = resolve_card_master_duel_id(
        card,
        real_mapping,
    )

    assert resolved is True
    assert card.master_duel_id == 7128

    print(card)

Card(name='"A" Cell Breeding Device', ydk_id=34541863, source_rarity='Common', master_duel_id=7128, rarity=<Rarity.N: 'N'>)
